In [ ]:
from datasets import load_dataset

dataset = load_dataset("jquigl/imdb-genres")

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_data = dataset["train"]
print(train_data[0])

{'movie title - year': 'Flaming Ears - 1992', 'genre': 'Fantasy', 'expanded-genres': 'Fantasy, Sci-Fi', 'rating': 6.0, 'description': 'Flaming Ears is a pop sci-fi lesbian fantasy feature set in the year 2700 in the fictive burned-out city of Asche. It follows the tangled lives of three women - Volley, Nun and Spy.'}


In [ ]:
import pandas as pd

df = train_data.to_pandas()
df.head()

,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."
3,Gulliver Returns - 2021,Fantasy,"Animation, Adventure, Family",4.4,The legendary Gulliver returns to the Kingdom ...
4,Prithvi Vallabh - 1924,Biography,"Biography, Drama, Romance",NaN,"Seminal silent historical film, the story feat..."


In [ ]:
int(df["rating"].isnull().sum())

69721

In [ ]:
df.shape

(238256, 5)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238256 entries, 0 to 238255
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movie title - year  238256 non-null  object 
 1   genre               238256 non-null  object 
 2   expanded-genres     238256 non-null  object 
 3   rating              168535 non-null  float64
 4   description         238256 non-null  object 
dtypes: float64(1), object(4)
memory usage: 9.1+ MB


In [ ]:
(df['rating']=='NaN').sum()

np.int64(0)

In [ ]:
df.isnull().sum()

movie title - year        0
genre                     0
expanded-genres           0
rating                69721
description               0
dtype: int64

In [ ]:
df.duplicated().sum()

np.int64(18)

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 238238 entries, 0 to 238255
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movie title - year  238238 non-null  object 
 1   genre               238238 non-null  object 
 2   expanded-genres     238238 non-null  object 
 3   rating              168533 non-null  float64
 4   description         238238 non-null  object 
dtypes: float64(1), object(4)
memory usage: 10.9+ MB


In [ ]:
df.isnull().sum()

movie title - year        0
genre                     0
expanded-genres           0
rating                69705
description               0
dtype: int64

In [ ]:
df.head(3)

,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."


In [ ]:
df['title'] = df['movie title - year'].str.split(' -').str[0]
df['year'] = df['movie title - year'].str.split(' -').str[1]

In [ ]:
df['tags'] = df['description'] + df['genre'] + df['expanded-genres']
df.head(1)

,movie title - year,genre,expanded-genres,rating,description,title,year,tags
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...,Flaming Ears,1992,Flaming Ears is a pop sci-fi lesbian fantasy f...


In [ ]:
movies = df[['title','tags', 'rating', 'year' ]]
movies.head(2)

,title,tags,rating,year
0,Flaming Ears,Flaming Ears is a pop sci-fi lesbian fantasy f...,6.0,1992
1,Jeg elsker dig,Six people - three couples - meet at random at...,5.8,1957


In [ ]:
movies['tags'][1]

'Six people - three couples - meet at random at a dance restaurant in the Copenhagen nightlife. A marriage swindler and former actress, an elderly Supreme Court attorney with wife as well as...                See full summary\xa0»RomanceComedy, Drama, Romance'

In [ ]:
# movies['tags'] = movies['tags'].apply(lambda x:x.lower())
# movies['tags'] = movies['tags'].fillna("").str.lower()
movies.loc[:, 'tags'] = movies['tags'].fillna("").str.lower()

In [ ]:
movies.loc[:, 'tags'] = movies['tags'].str.replace(r'[^\w\s]', '', regex=True)

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

movies = movies.copy()

movies['tags'] = (
    movies['tags']
    .fillna("")
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)   # remove punctuation
    .str.replace(r'\s+', ' ', regex=True)      # remove extra spaces
    .apply(lambda x: " ".join([w for w in x.split() if w not in stop_words]))
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Data Preprocessing

In [ ]:
# from sklearn.feature_extraction.text import CountVectorizer

# cv = CountVectorizer(
#     max_features=20000,
#     min_df=5,
#     max_df=0.8
# )

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(
    max_features=15000,
    min_df=5,
    stop_words='english'
)

In [ ]:
vectors = tf.fit_transform(movies['tags'])

In [ ]:
vectors.shape

(238238, 15000)

In [ ]:
tf.get_feature_names_out()

array(['000', '10', '100', ..., 'zorro', 'zoya', 'zurich'],
      shape=(15000,), dtype=object)

Stemming

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y = []
    
    for i in text.split():
        ps.stem(i)        
    
    return " ".join(y)

In [ ]:
# movies['tags'].apply(stem)

In [ ]:
# tf.get_feature_names_out()[1000:1500]

In [ ]:
movies.head(3)

,title,tags,rating,year
0,Flaming Ears,flaming ears pop scifi lesbian fantasy feature...,6.0,1992
1,Jeg elsker dig,six people three couples meet random dance res...,5.8,1957
2,Povjerenje,small unnamed town year 2025 krsto works agenc...,NaN,2021


Cosine Similarity

In [ ]:
# Start from 0
movies['movie_id'] = range(len(movies))

# OR start from 1
# movies['movie_id'] = range(1, len(movies)+1)

In [ ]:
movies[['movie_id', 'title', 'tags']].head(3)

,movie_id,title,tags
0,0,Flaming Ears,flaming ears pop scifi lesbian fantasy feature...
1,1,Jeg elsker dig,six people three couples meet random dance res...
2,2,Povjerenje,small unnamed town year 2025 krsto works agenc...


In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity

# def recommend_movies(input_text, movies, vectors, tfidf, top_k=5):
#     """
#     input_text: str (movie title or any keywords)
#     movies: dataframe
#     vectors: TF-IDF matrix of tags
#     tfidf: fitted TF-IDF vectorizer
#     top_k: number of movies to recommend
#     """
    
#     # Try to match input_text to a movie title
#     matched = movies[movies['title'].str.lower() == input_text.lower()]
    
#     if len(matched) > 0:
#         # Found exact movie title → get its index
#         movie_idx = matched.index[0]
#         input_vector = vectors[movie_idx]
#     else:
#         # Treat input_text as keywords → convert to TF-IDF
#         input_vector = tfidf.transform([input_text])
    
#     # Compute similarity with all movies
#     sim_scores = cosine_similarity(input_vector, vectors).flatten()
    
#     # Get top indices (ignore the input movie if it exists)
#     top_indices = sim_scores.argsort()[-top_k-1:-1][::-1]
    
#     # Return movie titles
#     return movies.iloc[top_indices]['title'].values

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
# def recommend_movies(input_text, top_k=5):
#     """
#     Recommend top_k movies similar to input_text.
#     input_text: movie title or any keywords
#     Returns: list of movie titles
#     """
#     # Try to match input_text to a movie title
#     matched = movies[movies['title'].str.lower() == input_text.lower()]
    
#     if len(matched) > 0:
#         # Exact movie title found → get its index
#         movie_idx = matched.index[0]
#         input_vector = vectors[movie_idx]
#     else:
#         # Treat input_text as keywords → convert to TF-IDF vector
#         input_vector = tf.transform([input_text])
    
#     # Compute cosine similarity with all movies
#     sim_scores = cosine_similarity(input_vector, vectors).flatten()
    
#     # Get top indices (ignore the input movie if it exists)
#     top_indices = sim_scores.argsort()[-top_k-1:-1][::-1]
    
#     # Return movie titles
#     return movies.iloc[top_indices]['title'].values

In [ ]:
# def recommend_movies(input_text, top_k=5):
#     """
#     Recommend top_k movies similar to input_text.
#     input_text: movie title or any keywords
#     Returns: list of movie titles
#     """
#     # 1️⃣ Try to match input_text to a movie title
#     matched = movies[movies['title'].str.lower() == input_text.lower()]
    
#     if len(matched) > 0:
#         # Exact movie title found → get its index
#         movie_idx = matched.index[0]
#         input_vector = vectors[movie_idx]
#         ignore_idx = [movie_idx]  # index to ignore in recommendations
#     else:
#         # Treat input_text as keywords → convert to TF-IDF vector
#         input_vector = tf.transform([input_text])
#         ignore_idx = []  # no index to ignore
    
#     # 2️⃣ Compute cosine similarity with all movies
#     sim_scores = cosine_similarity(input_vector, vectors).flatten()
    
#     # 3️⃣ Sort indices from high → low
#     sorted_indices = np.argsort(-sim_scores)  # minus sign reverses automatically
    
#     # 4️⃣ Remove ignored index(es)
#     sorted_indices = [i for i in sorted_indices if i not in ignore_idx]
    
#     # 5️⃣ Take top_k unique movies
#     unique_indices = []
#     for idx in sorted_indices:
#         if movies.iloc[idx]['title'] not in unique_indices:
#             unique_indices.append(movies.iloc[idx]['title'])
#         if len(unique_indices) >= top_k:
#             break
    
#     return np.array(unique_indices)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_movies(input_text, top_k=5):
    """
    Recommend top_k movies similar to input_text.
    input_text: movie title or any keywords
    Returns: list of movie titles
    """
    input_text_lower = input_text.lower()
    
    # 1️⃣ Try to find exact title match
    matched = movies[movies['title'].str.lower() == input_text_lower]
    
    if not matched.empty:
        # Exact title found → use its vector
        movie_idx = matched.index[0]
        input_vector = vectors[movie_idx]
        ignore_idx = {movie_idx}
    else:
        # Treat input as keywords → convert to TF-IDF
        input_vector = tfidf.transform([input_text])
        ignore_idx = set()
    
    # 2️⃣ Compute cosine similarity with all movie vectors
    sim_scores = cosine_similarity(input_vector, vectors).flatten()
    
    # 3️⃣ Get top indices sorted by similarity
    sorted_indices = np.argsort(-sim_scores)
    
    # 4️⃣ Filter out ignored indices & duplicates efficiently
    recommended = []
    seen_titles = set()
    for idx in sorted_indices:
        title = movies.iloc[idx]['title']
        if idx not in ignore_idx and title not in seen_titles:
            recommended.append(title)
            seen_titles.add(title)
        if len(recommended) >= top_k:
            break
    
    return recommended

In [ ]:
recommend_movies('korean')

NotFittedError: The TF-IDF vectorizer is not fitted